# External Dataset and Model Usage Example

This notebook demonstrates how to use dataset and model classes **defined outside of the Hyrax package** within Hyrax's training pipeline. It covers:

- Loading the [Galaxy10 dataset](https://astronn.readthedocs.io/en/stable/galaxy10sdss.html) via a custom `HyraxDataset` subclass (`Galaxy10Dataset`)
- Training Hyrax's built-in `HyraxCNN` model on that dataset
- Training a locally-defined `VGG11` model (registered with Hyrax) on the same dataset
- Comparing training loss between the two models

## Download the Galaxy10 Dataset

The [Galaxy10 dataset](https://astronn.readthedocs.io/en/stable/galaxy10sdss.html) from [AstroNN](https://github.com/henrysky/astroNN) contains ~22k labeled galaxy images across 10 morphological classes. It serves as a stand-in for any external astronomical image dataset.

The ~200MB HDF5 file will be downloaded and cached locally via `pooch`.

In [1]:
import pooch

file_path = pooch.retrieve(
    url="http://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5",
    known_hash="sha256:969A6B1CEFCC36E09FFFA86FEBD2F699A4AA19B837BA0427F01B0BC6DED458AF",
    fname="Galaxy10.h5",
    path="./data/galaxy_10",
)

In [ ]:
from hyrax import Hyrax

h = Hyrax()

## Configure Hyrax

Next we build a `data_request` dictionary that tells Hyrax which dataset class to use, where the data lives, and how to split it between training (80%) and validation (20%).

The key here is `dataset_class` — instead of a built-in Hyrax dataset, we point it at our locally-defined `Galaxy10Dataset` class. Hyrax will import and instantiate it automatically.

We also set `channels_first: True` in the dataset config so the images are provided in `(C, H, W)` order as expected by PyTorch.

In [ ]:
data_request = {
    "train": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
            "dataset_config": {
                "external_hyrax_example": {
                    "galaxy10_dataset": {
                        "channels_first": True,
                    },
                },
            },
        },
    },
    "validate": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
            "dataset_config": {
                "external_hyrax_example": {
                    "galaxy10_dataset": {
                        "channels_first": True,
                    },
                },
            },
        },
    },
}

h.set_config("data_request", data_request)

## Train HyraxCNN

We'll start by training `HyraxCNN`, a lightweight CNN that ships with Hyrax. We set the model name in config and then call `h.prepare()` to instantiate the data loaders.

In [ ]:
h.set_config("model.name", "HyraxCNN")

HyraxCNN's default `prepare_inputs` expects images in `(C, H, W)` format with standard normalization. Because Galaxy10 images are already in `(C, H, W)` order (we set `channels_first: True` earlier), we override `prepare_inputs` to apply the correct normalization and extract the label — replacing the default implementation before training begins.

In [ ]:
@staticmethod
def prepare_inputs(data_dict):
    import numpy as np

    if "data" not in data_dict:
        raise RuntimeError("Unable to find `data` key in data_dict")

    data = data_dict["data"]
    image = np.asarray(data["image"], dtype=np.float32)

    # normalize the image to have mean 0.5 and std 0.5
    image = np.asarray(image, dtype=np.float32)
    mean = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)
    std = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)

    mean = mean[:, None, None]
    std = std[:, None, None]

    normalized_image = (image - mean) / (std + 1e-8)

    label = None
    if "label" in data:
        label = np.asarray(data["label"], dtype=np.int64)

    return (normalized_image, label)

In [ ]:
model = h.model()
model.prepare_inputs = prepare_inputs

With the model configured, we kick off training. Hyrax handles the training loop, validation, and result logging automatically.

In [ ]:
model = h.train()

## Train VGG11

Now we switch to a locally-defined `VGG11` model. This is the key demonstration of using an **externally defined model** with Hyrax — the model is declared in `external_hyrax_example.models.vgg11` and registered with Hyrax via the `@hyrax_model` decorator, so we can reference it by its fully-qualified class path.

We reuse the same `prepare_inputs` override from easier comparison of the results.

In [ ]:
h.set_config("model.name", "external_hyrax_example.models.vgg11.VGG11")

In [ ]:
model = h.model()
model.prepare_inputs = prepare_inputs

Train VGG11 using the same data request as before.

In [ ]:
model = h.train()

## Results

Below is a comparison of training loss between the two models over the same number of epochs. VGG11 is a significantly deeper network and converges to a lower loss on this dataset.

![loss_values](../_static/HyraxCNN_vs_VGG11.png)

*Orange = HyraxCNN, Green = VGG11*